# 01 · Khám phá dữ liệu (Data Exploration)

## Mục tiêu
Hiểu **cấu trúc** và **chất lượng** của dataset Parkinson trước khi tiền xử lý / mô hình hoá:
kích thước, tên cột, kiểu dữ liệu (dtype), cột nhãn, số mẫu theo nhãn, thống kê mô tả (min/max/mean),
và kiểm tra dữ liệu thiếu / trùng / bất thường.

## Yêu cầu bài tập liên quan
- Xác minh cấu trúc & dtype (M02-T1), phân nhóm đặc trưng (M02-T2), phát hiện vấn đề (M02-T3),
  thống kê mô tả (M03-T1), kiểm tra chất lượng (M03-T2), báo cáo chất lượng (M03-T3).

## Phạm vi của notebook này
- **M02** (khám phá cấu trúc/dtype) + **M03** (thống kê mô tả + chất lượng).

## KHÔNG thuộc phạm vi
- Tiền xử lý / PCA (→ notebook 02), phân lớp (→ 03), phân cụm (→ 04), tổng hợp (→ 05).


## Phân công (theo `TASK_ASSIGNMENTS.md`)

| Nhiệm vụ | Phụ trách | Reviewer |
|---|---|---|
| M02-T1 Khám phá cấu trúc & dtype | M02 | M03 |
| M02-T2 Bản đồ nhóm đặc trưng | M02 | M03 |
| M02-T3 Phát hiện vấn đề (scale/outlier) | M02 | M04 |
| M03-T1 Thống kê mô tả | M03 | M02 |
| M03-T2 Kiểm tra chất lượng | M03 | M04 |
| M03-T3 Báo cáo chất lượng (gate M01) | M03 | M01 |

**Tên / MSSV thành viên:** M02 = **Lê Thiên Lộc** (MSSV `079306040024`) · M03 = **Huỳnh Gia Hợp** (MSSV `054205008367`) (xem `TEAM.md`).


## Điều kiện bắt đầu (đọc kỹ trước khi làm)

- **Đầu vào:** canonical CSV `data/processed/parkinsons_v1.csv` (do **M01-T2** tạo, xác nhận bởi **S-T1**).
  Dữ liệu chuẩn hoá: **22 feature + cột `status`**, đã loại cột `name` (xem `DATA_DICTIONARY.md`).
- **Nếu CSV chuẩn CHƯA tồn tại** → **DỪNG**, đánh dấu **`Blocked`** trong `TASK_BOARD.md`
  (chờ M01-T2 / S-T1). **KHÔNG tự tạo CSV** ở notebook này (vi phạm quy ước `data/README.md`).
- **Điều kiện chạy:** chạy từ **gốc project** (đường dẫn tương đối), không hardcode đường dẫn máy cá nhân.


In [1]:

from pathlib import Path

# Tìm canonical CSV từ thư mục hiện tại hoặc thư mục cha
project_root = Path.cwd()

if not (project_root / "data/processed/parkinsons_v1.csv").is_file():
    project_root = project_root.parent

CSV = project_root / "data/processed/parkinsons_v1.csv"

if not CSV.is_file():
    raise FileNotFoundError(
        f"[BLOCKED] Không tìm thấy canonical CSV: {CSV}. "
        "Kiểm tra lại dữ liệu đầu vào trước khi chạy Notebook."
    )

CSV = str(CSV)
print("[READY] Đã tìm thấy canonical CSV:", CSV)


[READY] Đã tìm thấy canonical CSV: /Users/hop/Nhom_3_KhaiThacDuLieu_Parkinsons/data/processed/parkinsons_v1.csv


In [2]:

%pip install tabulate



[notice] A new release of pip is available: 26.0.1 -> 26.3
[notice] To update, run: /opt/homebrew/opt/python@3.11/bin/python3.11 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [3]:
import sys
import pandas as pd
import numpy as np

df = pd.read_csv(CSV)

df.head()
print("Python version:", sys.version)
print("Pandas version:", pd.__version__)
print("NumPy version:", np.__version__)


Python version: 3.11.15 (main, Mar  3 2026, 00:52:57) [Clang 17.0.0 (clang-1700.6.3.2)]
Pandas version: 3.0.3
NumPy version: 2.4.6


## Bước 1 · Kiểm tra nguồn dữ liệu
**Mục tiêu:** xác nhận đúng file / đường dẫn và file đọc được.
**Vì sao:** sai nguồn → mọi thống kê sau vô nghĩa. Canonical CSV là đầu vào DUY NHẤT.


In [4]:
# TODO (M02-T1): đọc canonical CSV, xác nhận file đúng
# import pandas as pd
# df = pd.read_csv("data/processed/parkinsons_v1.csv")   # chỉ khi CSV chuẩn đã sẵn sàng
# df.head()   # sinh viên tự xem 5 hàng đầu
pass

## Bước 2 · Kích thước dữ liệu
**Mục tiêu:** biết số dòng (mẫu) và số cột (thuộc tính) thực tế.
**Vì sao:** đối chiếu với số đã xác minh (canonical: 22 feature + `status` = 23 cột, ~195 dòng).
**Câu hỏi tự kiểm:** số dòng/cột có khớp kỳ vọng không? Nếu lệch → ghi nhận và tra nguyên nhân.


In [5]:
# M02-T1: kích thước thực tế
print(f"Số dòng, số cột: {df.shape}")
assert df.shape == (195, 23), f"Kích thước khác kỳ vọng: {df.shape}"

Số dòng, số cột: (195, 23)


## Bước 3 · Tên cột & kiểu dữ liệu
**Mục tiêu:** liệt kê tên cột và dtype từng cột.
**Vì sao:** nhóm feature liên tục (FREQ/JITTER/SHIMMER/NOISE/NONLIN) sẽ cần chuẩn hoá sau (notebook 02);
cột `status` phải là số nhị phân (0/1).


In [6]:
# M02-T1: tên cột, dtype và kiểm tra kiểu dữ liệu
print("Các cột:", list(df.columns))
print(df.dtypes)

features = [column for column in df.columns if column != "status"]

assert all(pd.api.types.is_numeric_dtype(df[column]) for column in features)
assert pd.api.types.is_integer_dtype(df["status"])
assert set(df["status"].unique()) == {0, 1}

print("Đã xác nhận 22 feature là số và status là nhãn nguyên nhị phân 0/1.")

Các cột: ['MDVP:Fo(Hz)', 'MDVP:Fhi(Hz)', 'MDVP:Flo(Hz)', 'MDVP:Jitter(%)', 'MDVP:Jitter(Abs)', 'MDVP:RAP', 'MDVP:PPQ', 'Jitter:DDP', 'MDVP:Shimmer', 'MDVP:Shimmer(dB)', 'Shimmer:APQ3', 'Shimmer:APQ5', 'MDVP:APQ', 'Shimmer:DDA', 'NHR', 'HNR', 'status', 'RPDE', 'DFA', 'spread1', 'spread2', 'D2', 'PPE']
MDVP:Fo(Hz)         float64
MDVP:Fhi(Hz)        float64
MDVP:Flo(Hz)        float64
MDVP:Jitter(%)      float64
MDVP:Jitter(Abs)    float64
MDVP:RAP            float64
MDVP:PPQ            float64
Jitter:DDP          float64
MDVP:Shimmer        float64
MDVP:Shimmer(dB)    float64
Shimmer:APQ3        float64
Shimmer:APQ5        float64
MDVP:APQ            float64
Shimmer:DDA         float64
NHR                 float64
HNR                 float64
status                int64
RPDE                float64
DFA                 float64
spread1             float64
spread2             float64
D2                  float64
PPE                 float64
dtype: object
Đã xác nhận 22 feature là số và status l

## Bước 4 · Xác định thuộc tính & cột nhãn (theo tài liệu ĐÃ XÁC MINH)
**Mục tiêu:** phân biệt **feature (22 cột)** vs **label `status`**; xác nhận **`name` đã được loại**.
**Nguồn:** `DATA_DICTIONARY.md` (đã xác minh từ dataset).
- **Nhóm feature:** FREQ (3) · JITTER (5) · SHIMMER (6) · NOISE: `NHR`,`HNR` · NONLIN: `RPDE`,`DFA`,`spread1`,`spread2`,`D2`,`PPE` (6).
- **Label:** `status` (`0`=khỏe, `1`=Parkinson).
**Câu hỏi tự kiểm:** canonical CSV có còn cột `name` không? (Phải **KHÔNG**.) Có đủ 22 feature + `status` không?


In [7]:
# M02-T1/T2: xác nhận schema và bản đồ nhóm đặc trưng
assert "name" not in df.columns, "LỖI: canonical CSV vẫn giữ cột 'name'"
assert "status" in df.columns, "LỖI: thiếu cột nhãn 'status'"
assert len(features) == 22, f"Số feature phải là 22, nhận được {len(features)}"

feature_groups = {
    "FREQ": ["MDVP:Fo(Hz)", "MDVP:Fhi(Hz)", "MDVP:Flo(Hz)"],
    "JITTER": ["MDVP:Jitter(%)", "MDVP:Jitter(Abs)", "MDVP:RAP", "MDVP:PPQ", "Jitter:DDP"],
    "SHIMMER": ["MDVP:Shimmer", "MDVP:Shimmer(dB)", "Shimmer:APQ3", "Shimmer:APQ5", "MDVP:APQ", "Shimmer:DDA"],
    "NOISE": ["NHR", "HNR"],
    "NONLIN": ["RPDE", "DFA", "spread1", "spread2", "D2", "PPE"],
}

assert sorted(sum(feature_groups.values(), [])) == sorted(features)

for group, columns in feature_groups.items():
    print(f"{group} ({len(columns)}): {', '.join(columns)}")

FREQ (3): MDVP:Fo(Hz), MDVP:Fhi(Hz), MDVP:Flo(Hz)
JITTER (5): MDVP:Jitter(%), MDVP:Jitter(Abs), MDVP:RAP, MDVP:PPQ, Jitter:DDP
SHIMMER (6): MDVP:Shimmer, MDVP:Shimmer(dB), Shimmer:APQ3, Shimmer:APQ5, MDVP:APQ, Shimmer:DDA
NOISE (2): NHR, HNR
NONLIN (6): RPDE, DFA, spread1, spread2, D2, PPE


## Bước 5 · Số mẫu theo từng nhãn
**Mục tiêu:** đếm số mẫu ở mỗi giá trị `status`.
**Vì sao:** biết lớp có **cân bằng** không → ảnh hưởng cách chia train/test & chọn metric (notebook 03).
**Câu hỏi tự kiểm:** tỉ lệ 0/1 là bao nhiêu? Mất cân bằng có đáng kể không?


In [8]:
import pandas as pd

# Đọc canonical CSV của nhóm
df = pd.read_csv(CSV)
df.head()

# Kiểm tra dữ liệu
print("Kích thước dữ liệu:", df.shape)
print("Danh sách cột:", df.columns.tolist())
display(df.head())

Kích thước dữ liệu: (195, 23)
Danh sách cột: ['MDVP:Fo(Hz)', 'MDVP:Fhi(Hz)', 'MDVP:Flo(Hz)', 'MDVP:Jitter(%)', 'MDVP:Jitter(Abs)', 'MDVP:RAP', 'MDVP:PPQ', 'Jitter:DDP', 'MDVP:Shimmer', 'MDVP:Shimmer(dB)', 'Shimmer:APQ3', 'Shimmer:APQ5', 'MDVP:APQ', 'Shimmer:DDA', 'NHR', 'HNR', 'status', 'RPDE', 'DFA', 'spread1', 'spread2', 'D2', 'PPE']


,MDVP:Fo(Hz),MDVP:Fhi(Hz),MDVP:Flo(Hz),MDVP:Jitter(%),MDVP:Jitter(Abs),MDVP:RAP,MDVP:PPQ,Jitter:DDP,MDVP:Shimmer,MDVP:Shimmer(dB),...,Shimmer:DDA,NHR,HNR,status,RPDE,DFA,spread1,spread2,D2,PPE
0,119.992,157.302,74.997,0.00784,0.00007,0.00370,0.00554,0.01109,0.04374,0.426,...,0.06545,0.02211,21.033,1,0.414783,0.815285,-4.813031,0.266482,2.301442,0.284654
1,122.400,148.650,113.819,0.00968,0.00008,0.00465,0.00696,0.01394,0.06134,0.626,...,0.09403,0.01929,19.085,1,0.458359,0.819521,-4.075192,0.335590,2.486855,0.368674
2,116.682,131.111,111.555,0.01050,0.00009,0.00544,0.00781,0.01633,0.05233,0.482,...,0.08270,0.01309,20.651,1,0.429895,0.825288,-4.443179,0.311173,2.342259,0.332634
3,116.676,137.871,111.366,0.00997,0.00009,0.00502,0.00698,0.01505,0.05492,0.517,...,0.08771,0.01353,20.644,1,0.434969,0.819235,-4.117501,0.334147,2.405554,0.368975
4,116.014,141.781,110.655,0.01284,0.00011,0.00655,0.00908,0.01966,0.06425,0.584,...,0.10470,0.01767,19.649,1,0.417356,0.823484,-3.747787,0.234513,2.332180,0.410335


In [9]:
# M03-T1: thống kê số lượng và tỷ lệ mẫu theo nhãn
label_counts = df["status"].value_counts().sort_index()
label_percent = df["status"].value_counts(normalize=True).sort_index() * 100

label_summary = pd.DataFrame({
    "Số lượng": label_counts,
    "Tỷ lệ (%)": label_percent
})

display(label_summary.round(2))

print("Tổng số mẫu:", label_counts.sum())

assert label_counts.sum() == len(df)
assert set(label_counts.index) == {0, 1}

print("Đã kiểm tra phân bố nhãn thành công.")

,Số lượng,Tỷ lệ (%)
status,,
0,48,24.62
1,147,75.38


Tổng số mẫu: 195
Đã kiểm tra phân bố nhãn thành công.


## Bước 6 · min / max / mean cho thuộc tính số
**Mục tiêu:** thống kê mô tả các feature liên tục.
**Vì sao:** phát hiện chênh lệch scale (vd `MDVP:Jitter(Abs)` rất nhỏ, `MDVP:Shimmer` khác)
→ cơ sở để chuẩn hoá (notebook 02).
**Làm gì:** dùng `describe()` / `min()`,`max()`,`mean()`; so sánh **giữa các nhóm** (FREQ vs JITTER vs NONLIN).


In [10]:
# M03-T1: Thống kê mô tả các thuộc tính số

# Lấy các cột số, loại bỏ status vì đây là nhãn phân lớp
numeric_features = df.drop(columns=["status"], errors="ignore").select_dtypes(include="number").drop(
    columns=["status"],
    errors="ignore"
)

# Tính các chỉ số thống kê
stats = pd.DataFrame({
    "mean": numeric_features.mean(),
    "median": numeric_features.median(),
    "std": numeric_features.std(),
    "min": numeric_features.min(),
    "max": numeric_features.max()
})

print("BẢNG THỐNG KÊ MÔ TẢ:")
display(stats.round(4))

BẢNG THỐNG KÊ MÔ TẢ:


,mean,median,std,min,max
MDVP:Fo(Hz),154.2286,148.7900,41.3901,88.3330,260.1050
MDVP:Fhi(Hz),197.1049,175.8290,91.4915,102.1450,592.0300
MDVP:Flo(Hz),116.3246,104.3150,43.5214,65.4760,239.1700
MDVP:Jitter(%),0.0062,0.0049,0.0048,0.0017,0.0332
MDVP:Jitter(Abs),0.0000,0.0000,0.0000,0.0000,0.0003
MDVP:RAP,0.0033,0.0025,0.0030,0.0007,0.0214
MDVP:PPQ,0.0034,0.0027,0.0028,0.0009,0.0196
Jitter:DDP,0.0099,0.0075,0.0089,0.0020,0.0643
MDVP:Shimmer,0.0297,0.0230,0.0189,0.0095,0.1191
MDVP:Shimmer(dB),0.2823,0.2210,0.1949,0.0850,1.3020


In [11]:
# M03-T1: Thống kê min/max/mean theo từng feature
# Mỗi feature một hàng
# M03-T1: Thống kê mô tả min/max/mean và skew/kurtosis theo từng feature

stats = numeric_features.describe().T

# Bổ sung độ lệch và độ nhọn
stats["skew"] = numeric_features.skew()
stats["kurtosis"] = numeric_features.kurt()

# Hiển thị kết quả
display(stats.round(4))

,count,mean,std,min,25%,50%,75%,max,skew,kurtosis
MDVP:Fo(Hz),195.0,154.2286,41.3901,88.3330,117.5720,148.7900,182.7690,260.1050,0.5917,-0.6279
MDVP:Fhi(Hz),195.0,197.1049,91.4915,102.1450,134.8625,175.8290,224.2055,592.0300,2.5421,7.6272
MDVP:Flo(Hz),195.0,116.3246,43.5214,65.4760,84.2910,104.3150,140.0185,239.1700,1.2174,0.6546
MDVP:Jitter(%),195.0,0.0062,0.0048,0.0017,0.0035,0.0049,0.0074,0.0332,3.0849,12.0309
MDVP:Jitter(Abs),195.0,0.0000,0.0000,0.0000,0.0000,0.0000,0.0001,0.0003,2.6491,10.8690
MDVP:RAP,195.0,0.0033,0.0030,0.0007,0.0017,0.0025,0.0038,0.0214,3.3607,14.2138
MDVP:PPQ,195.0,0.0034,0.0028,0.0009,0.0019,0.0027,0.0040,0.0196,3.0739,11.9639
Jitter:DDP,195.0,0.0099,0.0089,0.0020,0.0050,0.0075,0.0115,0.0643,3.3621,14.2248
MDVP:Shimmer,195.0,0.0297,0.0189,0.0095,0.0165,0.0230,0.0379,0.1191,1.6665,3.2383
MDVP:Shimmer(dB),195.0,0.2823,0.1949,0.0850,0.1485,0.2210,0.3500,1.3020,1.9994,5.1282


In [12]:

# M03-T1: Xuất đầy đủ 7 chỉ số thống kê mô tả dạng Markdown

stats_report = pd.DataFrame({
    "mean": numeric_features.mean(),
    "median": numeric_features.median(),
    "std": numeric_features.std(),
    "min": numeric_features.min(),
    "max": numeric_features.max(),
    "skew": numeric_features.skew(),
    "kurtosis": numeric_features.kurt()
})

stats_report.index.name = "Feature"

print(stats_report.round(4).to_markdown())


| Feature          |     mean |   median |     std |      min |      max |    skew |   kurtosis |
|:-----------------|---------:|---------:|--------:|---------:|---------:|--------:|-----------:|
| MDVP:Fo(Hz)      | 154.229  | 148.79   | 41.3901 |  88.333  | 260.105  |  0.5917 |    -0.6279 |
| MDVP:Fhi(Hz)     | 197.105  | 175.829  | 91.4915 | 102.145  | 592.03   |  2.5421 |     7.6272 |
| MDVP:Flo(Hz)     | 116.325  | 104.315  | 43.5214 |  65.476  | 239.17   |  1.2174 |     0.6546 |
| MDVP:Jitter(%)   |   0.0062 |   0.0049 |  0.0048 |   0.0017 |   0.0332 |  3.0849 |    12.0309 |
| MDVP:Jitter(Abs) |   0      |   0      |  0      |   0      |   0.0003 |  2.6491 |    10.869  |
| MDVP:RAP         |   0.0033 |   0.0025 |  0.003  |   0.0007 |   0.0214 |  3.3607 |    14.2138 |
| MDVP:PPQ         |   0.0034 |   0.0027 |  0.0028 |   0.0009 |   0.0196 |  3.0739 |    11.9639 |
| Jitter:DDP       |   0.0099 |   0.0075 |  0.0089 |   0.002  |   0.0643 |  3.3621 |    14.2248 |
| MDVP:Shimmer     |

In [13]:
# M03-T2: Kiểm tra outlier bằng phương pháp IQR

# Tính Q1, Q3 và IQR
Q1 = numeric_features.quantile(0.25)
Q3 = numeric_features.quantile(0.75)
IQR = Q3 - Q1

# Xác định ngưỡng dưới và ngưỡng trên
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

# Xác định các giá trị outlier
outlier_mask = (
    (numeric_features.lt(lower_bound)) |
    (numeric_features.gt(upper_bound))
)

# Thống kê số lượng outlier theo từng feature
outlier_summary = pd.DataFrame({
    "Q1": Q1,
    "Q3": Q3,
    "IQR": IQR,
    "Lower_bound": lower_bound,
    "Upper_bound": upper_bound,
    "Outlier_count": outlier_mask.sum(),
    "Outlier_percent": outlier_mask.mean() * 100
})

print("THỐNG KÊ OUTLIER THEO FEATURE:")
display(outlier_summary.round(4))

# Tổng hợp số feature và số dòng có outlier
features_with_outlier = (outlier_mask.sum() > 0).sum()
rows_with_outlier = outlier_mask.any(axis=1).sum()

print("\nTỔNG KẾT:")
print("Tổng số feature:", numeric_features.shape[1])
print("Số feature có outlier:", features_with_outlier)
print("Tổng số dòng:", len(numeric_features))
print("Số dòng có ít nhất một outlier:", rows_with_outlier)
print("Tỷ lệ dòng có outlier:", round(rows_with_outlier / len(numeric_features) * 100, 2), "%")

THỐNG KÊ OUTLIER THEO FEATURE:


,Q1,Q3,IQR,Lower_bound,Upper_bound,Outlier_count,Outlier_percent
MDVP:Fo(Hz),117.5720,182.7690,65.1970,19.7765,280.5645,0,0.0000
MDVP:Fhi(Hz),134.8625,224.2055,89.3430,0.8480,358.2200,11,5.6410
MDVP:Flo(Hz),84.2910,140.0185,55.7275,0.6997,223.6098,9,4.6154
MDVP:Jitter(%),0.0035,0.0074,0.0039,-0.0024,0.0132,14,7.1795
MDVP:Jitter(Abs),0.0000,0.0001,0.0000,-0.0000,0.0001,7,3.5897
MDVP:RAP,0.0017,0.0038,0.0022,-0.0016,0.0071,14,7.1795
MDVP:PPQ,0.0019,0.0040,0.0021,-0.0013,0.0071,15,7.6923
Jitter:DDP,0.0050,0.0115,0.0065,-0.0048,0.0213,14,7.1795
MDVP:Shimmer,0.0165,0.0379,0.0214,-0.0156,0.0700,8,4.1026
MDVP:Shimmer(dB),0.1485,0.3500,0.2015,-0.1538,0.6523,10,5.1282



TỔNG KẾT:
Tổng số feature: 22
Số feature có outlier: 19
Tổng số dòng: 195
Số dòng có ít nhất một outlier: 47
Tỷ lệ dòng có outlier: 24.1 %


In [18]:

# M03-T2: Xuất bảng IQR dạng Markdown

outlier_summary.index.name = "Feature"

print(outlier_summary.round(4).to_markdown())

# Tổng hợp số lượng candidate outlier
print("\n===== TONG KET IQR =====")
print("Tổng số feature:", numeric_features.shape[1])
print("Feature có outlier:", (outlier_mask.sum() > 0).sum())
print("Feature không có outlier:", (outlier_mask.sum() == 0).sum())
print("Tổng số mẫu:", len(numeric_features))
print("Số mẫu có ít nhất một outlier:", outlier_mask.any(axis=1).sum())
print("Tỷ lệ:", round(outlier_mask.any(axis=1).mean() * 100, 2), "%")

# M03-T2: Xuất toàn bộ bảng IQR ra file Markdown

from pathlib import Path

# Xác định thư mục gốc dự án
project_root = Path.cwd()

if not (project_root / "results").is_dir():
    project_root = project_root.parent

# Kiểm tra thư mục results
results_dir = project_root / "results"
results_dir.mkdir(parents=True, exist_ok=True)

# Xuất bảng IQR
iqr_markdown = outlier_summary.round(4).to_markdown()

output_path = results_dir / "iqr_table.md"
output_path.write_text(iqr_markdown + "\n", encoding="utf-8")

print("Đã xuất bảng IQR:", )
print("Số feature trong bảng:", len(outlier_summary))



| Feature          |       Q1 |       Q3 |     IQR |   Lower_bound |   Upper_bound |   Outlier_count |   Outlier_percent |
|:-----------------|---------:|---------:|--------:|--------------:|--------------:|----------------:|------------------:|
| MDVP:Fo(Hz)      | 117.572  | 182.769  | 65.197  |       19.7765 |      280.565  |               0 |            0      |
| MDVP:Fhi(Hz)     | 134.863  | 224.206  | 89.343  |        0.848  |      358.22   |              11 |            5.641  |
| MDVP:Flo(Hz)     |  84.291  | 140.018  | 55.7275 |        0.6997 |      223.61   |               9 |            4.6154 |
| MDVP:Jitter(%)   |   0.0035 |   0.0074 |  0.0039 |       -0.0024 |        0.0132 |              14 |            7.1795 |
| MDVP:Jitter(Abs) |   0      |   0.0001 |  0      |       -0      |        0.0001 |               7 |            3.5897 |
| MDVP:RAP         |   0.0017 |   0.0038 |  0.0022 |       -0.0016 |        0.0071 |              14 |            7.1795 |
| MDVP:PPQ      

## Bước 7 · Dữ liệu thiếu, trùng lặp, bất thường
**Mục tiêu:** kiểm tra NaN, hàng trùng, giá trị ngoài khoảng (candidate outlier).
**Vì sao:** dữ liệu bẩn phải được xử lý TRƯỚC khi mô hình (đề xuất phương pháp cho M04).
**Câu hỏi tự kiểm:** có NaN không? Có hàng trùng không? Có giá trị rõ ràng bất thường không?


In [15]:
# M03-T2: Kiểm tra chất lượng dữ liệu

# 1. Kiểm tra số lượng giá trị thiếu theo từng cột
print("===== 1. DỮ LIỆU THIẾU (NaN) =====")
missing = df.isna().sum()
print(missing)
print("Tổng số giá trị thiếu:", missing.sum())

# 2. Kiểm tra số dòng trùng lặp
print("\n===== 2. DỮ LIỆU TRÙNG LẶP =====")
duplicate_count = df.duplicated().sum()
print("Số dòng bị trùng:", duplicate_count)

# 3. Thống kê các phân vị để tìm giá trị bất thường
print("\n===== 3. THỐNG KÊ PHÂN VỊ 1% VÀ 99% =====")
numeric_features = df.select_dtypes(include="number").drop(
    columns=["status"],
    errors="ignore"
)

display(
    numeric_features.describe(
        percentiles=[0.01, 0.25, 0.5, 0.75, 0.99]
    ).T.round(4)
)

===== 1. DỮ LIỆU THIẾU (NaN) =====
MDVP:Fo(Hz)         0
MDVP:Fhi(Hz)        0
MDVP:Flo(Hz)        0
MDVP:Jitter(%)      0
MDVP:Jitter(Abs)    0
MDVP:RAP            0
MDVP:PPQ            0
Jitter:DDP          0
MDVP:Shimmer        0
MDVP:Shimmer(dB)    0
Shimmer:APQ3        0
Shimmer:APQ5        0
MDVP:APQ            0
Shimmer:DDA         0
NHR                 0
HNR                 0
status              0
RPDE                0
DFA                 0
spread1             0
spread2             0
D2                  0
PPE                 0
dtype: int64
Tổng số giá trị thiếu: 0

===== 2. DỮ LIỆU TRÙNG LẶP =====
Số dòng bị trùng: 0

===== 3. THỐNG KÊ PHÂN VỊ 1% VÀ 99% =====


,count,mean,std,min,1%,25%,50%,75%,99%,max
MDVP:Fo(Hz),195.0,154.2286,41.3901,88.3330,94.8669,117.5720,148.7900,182.7690,245.9267,260.1050
MDVP:Fhi(Hz),195.0,197.1049,91.4915,102.1450,107.3904,134.8625,175.8290,224.2055,586.6841,592.0300
MDVP:Flo(Hz),195.0,116.3246,43.5214,65.4760,65.7801,84.2910,104.3150,140.0185,232.7722,239.1700
MDVP:Jitter(%),195.0,0.0062,0.0048,0.0017,0.0018,0.0035,0.0049,0.0074,0.0302,0.0332
MDVP:Jitter(Abs),195.0,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0001,0.0002,0.0003
MDVP:RAP,195.0,0.0033,0.0030,0.0007,0.0008,0.0017,0.0025,0.0038,0.0180,0.0214
MDVP:PPQ,195.0,0.0034,0.0028,0.0009,0.0010,0.0019,0.0027,0.0040,0.0163,0.0196
Jitter:DDP,195.0,0.0099,0.0089,0.0020,0.0023,0.0050,0.0075,0.0115,0.0541,0.0643
MDVP:Shimmer,195.0,0.0297,0.0189,0.0095,0.0101,0.0165,0.0230,0.0379,0.0919,0.1191
MDVP:Shimmer(dB),195.0,0.2823,0.1949,0.0850,0.0888,0.1485,0.2210,0.3500,0.9353,1.3020


## Kết quả & nhận xét (sinh viên tự điền — KHÔNG điền sẵn số)

**Bảng ghi nhận (điền KHI CHẠY THẬT):**

| Hạng mục | Giá trị đo được | Nhận xét |
|---|---|---|
| Số dòng (mẫu) | _195_ | _Có 195 mẫu dữ liệu_ |
| Số cột (canonical) | _23_ | _Gồm 22 feature và 1 cột nhãn_ |
| Cột nhãn | `status` | 0=khỏe, 1=Parkinson |
| Số mẫu label=0 | _48_ | _Chiếm 24,62%_ |
| Số mẫu label=1 | _147_ | _Chiếm 75,38%_ |
| NaN (tổng) | _0_ | _Không có giá trị thiếu_ |
| Hàng trùng | _0_ | _Không phát hiện dòng trùng lặp_ |
| Feature có scale cực đoan | _Có_ | _Các feature có thang đo khác nhau đáng kể_ |

**Câu hỏi gợi ý (phân tích dựa trên kết quả thật):**
1. Dataset có đủ lớn / cân bằng để học phân lớp nhị phân không?

- Dataset có 195 mẫu, gồm 147 mẫu Parkinson và 48 mẫu khỏe. Dữ liệu có sự mất cân bằng giữa hai lớp, tỷ lệ khoảng 75,38% và 24,62%. Dataset tương đối nhỏ nên cần chú ý khi chia tập train/test. Nên sử dụng phân tầng (stratify) và đánh giá bằng Precision, Recall, F1-score, Confusion Matrix thay vì chỉ dựa vào Accuracy.

2. Nhóm feature nào cần chuẩn hoá trước khi PCA / mô hình?

- Các feature có đơn vị và thang đo khác nhau. Ví dụ, MDVP:Fo(Hz), MDVP:Fhi(Hz), MDVP:Flo(Hz) có giá trị ở mức hàng trăm, trong khi MDVP:Jitter(Abs), MDVP:Jitter(%), MDVP:RAP có giá trị rất nhỏ. Vì vậy, cần chuẩn hóa các feature số trước khi thực hiện PCA và huấn luyện mô hình.

3. Có xử lý nào cần làm trước khi đưa vào mô hình không?

- Cần chuẩn hóa các feature số, kiểm tra thêm các giá trị bất thường và chia tập dữ liệu có phân tầng theo nhãn status. Không phát hiện giá trị thiếu hoặc hàng trùng trong lần kiểm tra hiện tại nên chưa cần xử lý hai vấn đề này.


## Checklist (trạng thái ban đầu: `Todo`)

### Hoàn thành nhiệm vụ
- [x] Đã đọc canonical CSV (không CSV riêng).
- [x] Đã xác minh kích thước, tên cột, dtype.
- [x] Đã xác nhận `name` đã drop, `status` là nhãn, đủ 22 feature.
- [x] Đã thống kê số mẫu theo nhãn, min/max/mean.
- [x] Đã kiểm tra thiếu / trùng / bất thường.

### Kiểm tra chất lượng
- [x] Số liệu GHI LẠI từ kết quả chạy thật (không bịa).
- [x] Danh sách vấn đề rõ, đủ để M04 xử lý.

### Bàn giao
- [X] Đã viết `results/quality_report.md` (M03-T3) + ghi vào `results/`.
- [X] Đã chuyển M02-T1..T3, M03-T1..T3 sang `Review` trong `TASK_BOARD.md`.
- [X] Đã thông báo reviewer (M03 / M04 / M01-gate).

### Vấn đề chưa giải quyết (ghi rõ nếu có)
- [ ] Dữ liệu mất cân bằng nhãn: status=1 có 147 mẫu (75.38%), status=0 có 48 mẫu (24.62%). Các feature có thang đo khác nhau, cần xem xét chuẩn hóa trước PCA và huấn luyện mô hình.


## Tái lập & xử lý Blocked

- **Nguồn dữ liệu:** canonical CSV `data/processed/parkinsons_v1.csv` (version v1, do M01-T2 tạo).
- **Đầu vào:** canonical CSV + `DATA_DICTIONARY.md`. **Đầu ra dự kiến:** bảng khám phá + `results/quality_report.md`.
- **Không hardcode** đường dẫn máy cá nhân — dùng đường dẫn tương đối (chạy từ gốc project).
- **Ghi lại** (vào `results/`): seed nếu dùng, phiên bản thư viện, cách xử lý NaN/trùng (nếu chọn).
- **Nếu thiếu đầu vào / phụ thuộc chưa xong:** đánh dấu **`Blocked`** trong `TASK_BOARD.md` + ghi rõ
  thiếu gì, ai cung cấp; thông báo M01 (điều phối). Xem `HANDOFF.md` mục 6.
